In [1]:
import import_ipynb
from rco_test import *
from ast import *
from utils import *
from x86_ast import *


In [2]:
def select_arg(e: expr) -> arg:
        # YOUR CODE HERE
        match e:
            case Constant(value):
                return Immediate(value)
            case Name(var):
                return Variable(var)
         

In [3]:
def select_stmt(s: stmt) -> list[instr]:
        # YOUR CODE HERE
        match s:
          
            case Assign([Name(var)],UnaryOp(USub(), v)):
                val = select_arg(v)
                return [Instr('movq',[val, Reg('rax')]),
                        Instr('negq',[Reg('rax')]),
                        Instr('movq',[Reg('rax'), Variable(var)])]
            case Assign([Name(var)], BinOp(left,Add(), right)):
                l = select_arg(left)
                r = select_arg(right)
                if isinstance(left, Name) and left.id == var:
                     return [Instr('addq',[r,Variable(var)])]
                elif isinstance(right, Name) and right.id == var:
                     return [Instr('addq',[l,Variable(var)])]
                else:
                    return [Instr('movq',[l, Reg('rax')]),
                        Instr('addq',[r,Reg('rax')]),
                        Instr('movq',[Reg('rax'), Variable(var)])]
            case Assign([Name(var)], BinOp(left, Sub(), right)):
                l = select_arg(left)
                r = select_arg(right)
                if isinstance(left, Name) and left.id == var:
                     return [Instr('subq',[r,Variable(var)])]
                else:
                     return [Instr('movq',[l,Reg('rax')]),
                        Instr('subq',[r,Reg('rax')]),
                        Instr('movq', [Reg('rax'),Variable(var)])]
            case Assign([Name(var)], Call(Name('input_int'))):
                return [Callq('read_int',0),
                        Instr('movq', [Reg('rax'), Variable(var)])]
            case Assign([Name(var)],value):
                  new_value = select_arg(value)
                  return [Instr('movq',[new_value,Variable(var)])]
            case Expr(Call(Name('print'),[arg])):
                new_arg = select_arg(arg)
                return [Instr('movq',[new_arg, Reg('rdi')]), 
                        Callq('print_int',1)]
           

In [4]:
def select_instruction(p : Module) -> X86Program:
    match p:
        case Module(body):
            new_body = []
            for stmt in body:
                new_body.extend(select_stmt(stmt))
                
            return X86Program(new_body)

In [7]:
if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    a = 42
    b = a
    print(b)
    """)
    parsed_code = parse(code)
    rco_code = remove_complex_operands(parsed_code)
    select_instr_program = select_instruction(rco_code)
    for i in select_instr_program:
        print(i)
    

TypeError: 'X86Program' object is not iterable